# Chapter 6 of Book Hands on LLM

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [2]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [4]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False, # needs to be true if using top_p or temperature
)

Device set to use cuda


In [5]:
# Simple Prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


 Why did the chicken join the band? Because it had the drumsticks!


In [6]:
# how the list of dicts is transformed into a prompt template
# through the use of the method apply_chat_template
promt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(promt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


In [7]:
'''
using Temperature to create more diverse output; Remember,
a higher temperature increases the likelihood that less
probable tokens are generated and vice versa
'''
# low temperature
output = pipe(messages, do_sample=True, temperature=0.1)
print(output[0]["generated_text"])
# high temperature
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!
 Why did the chicken go to the church?


In [8]:
'''
using Top_p (aka nucleus sampling) to create more diverse output; Top_p
is a sampling technique that controls which subset of tokens (the nucleus) the
LLM can consider. It will consider tokens until it reaches their cumulative
probability; So if top_p is set to 0.1 the llm will consider tokens until it
reaches that value, if set to 1 it will consider all tokens
'''
# low top_p
output = pipe(messages, do_sample=True, top_p=0.1)
print(output[0]["generated_text"])
# high top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!
 Why do chickens make terrible comedians? Because they can't peck at punchlines!




---


# Creating Complex Prompt through Iteration

A **general prompt** can include the followng components: 1) an instruction; 2) data; and 3)output indicators.

A **complex prompt** can include the following components: 1) persona; 2) instruction; 3) context; 4) format; 5) audience; 6) tone; and 7) data


*   **Persona**: Describe what role the LLM should take on.
*   **Instruction**: The task itself. Make sure this is as specific as possible. We do not want to leave much room for interpretation.
*   **Context**: Additional information describing the context of the task
*   **Format**: The format the LLM should use to output the generated text
*   **Audience**: The target of the generated text. This also describes the level of the generated output.
*   **Tone**: The tone of voice the LLM should use in the generated text.
*   **Data**: The main data related to the task itself


---




In [9]:
# Creating a Complex prompt from Prompt Components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the idea\n"
tone = "The tone should be professional and clear.\n"
text = """Pamir was a four-masted barque built for the German shipping
          company F. Laeisz. One of their famous Flying P-Liners,
          she was the last commercial sailing ship to round Cape Horn,
          in 1949. By 1957, she had been outmoded by modern bulk carriers and could not operate at a profit.
          Her shipping consortium's inability to finance much-needed repairs or to recruit sufficient sail-trained officers caused severe technical difficulties.
          On 21 September 1957, she was caught in Hurricane Carrie and sank off the Azores, with only six survivors rescued after an extensive search"""
data = f"Text to summarize: {text}"

In [10]:
# Iteration one of prompt components
query = instruction + context + data
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

.

Summary: Pamir, a four-masted barque and one of the Flying P-Liners, was the last commercial sailing ship to round Cape Horn in 1949. By 1957, she was outdated and unprofitable due to the inability of her shipping consortium to finance repairs or recruit sail-trained officers. On 21 September 1957, Pamir was caught in Hurricane Carrie and sank off the Azores, with only six survivors rescued.





In [11]:
# Iteration two of prompt components
query = persona+instruction + context + data
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

.

Summary: Pamir, a four-masted barque built by the German shipping company F. Laeisz, was the last commercial sailing ship to round Cape Horn in 1949. By 1957, she was outmoded by modern bulk carriers and could not operate profitably. The shipping consortium's inability to finance repairs or recruit sail-trained officers led to severe technical difficulties. Pamir sank off the Azores on 21 September 1957 during Hurricane Carrie, with only six survivors rescued after an extensive search.


Text to summarize: The paper discusses the impact of climate change on the frequency and intensity of hurricanes in the Atlantic Ocean. It presents a comprehensive analysis of historical hurricane data from the past century, highlighting a significant increase in both the number and severity of hurricanes over the last few decades. The authors attribute this trend to rising sea surface temperatures and changes in atmospheric conditions due to global warming. The paper also explores the potential imp

In [12]:
# Iteration two of prompt components
query = persona+instruction + context + data + data_format
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])


Method:
- Analyze the historical data of Pamir's voyages and operational challenges.
- Examine the financial records of the shipping consortium to understand the economic pressures.
- Review the technical logs and crew reports to identify the causes of the ship's difficulties.
- Investigate meteorological data to assess the impact of Hurricane Carrie on Pamir.
- Compile survivor accounts and search operation records to evaluate the rescue efforts.

Results:
The paper reveals that Pamir, a renowned four-masted barque, faced a series of challenges that led to her sinking in 1957. The analysis of her operational history shows that the ship was outdated by the time of her final voyage, struggling to compete with modern bulk carriers. Financial difficulties plagued the shipping consortium, preventing necessary repairs and the recruitment of skilled sailors. Technical issues were exacerbated by the ship's age and lack of maintenance. The catastrophic encounter with Hurricane Carrie was the 

# In-Context Learning: Providing Examples

For in-context learning instead of describing the task we show it the task by
giving it correct examples. The number of examples determines the type of in context prompting

*   **One-shot prompt**: prompting with a single example
*   **Few-shot prompt**: prompting with more than one example

This method was invented in the paper [Tom Brown et al. “Language models are few-shot learners.” Advances in Neural Information Processing Systems 33 (2020): 1877–1901.](https://arxiv.org/pdf/2005.14165)

To do so in the prompt we have to differentiate between our question in our list of dicts with the key **user** and the answer we want the model to produce with the key **assistant**

---



In [13]:
'''
In this example the goal of the prompt is to generate a
sentence with a made-up word using one-shot prompting
'''
# A single example of using a made-up word in a sentence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
# seeing what the above one shot prompt looks like as a template
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [15]:
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

 During the medieval reenactment, the knight skillfully screeged the wooden shield to demonstrate his prowess in combat.


# Chain Prompting: Breaking up the Problem

Chain prompting is a technique that takes the **output** of one prompt and uses it as **input** into the next prompt. This is done by calling  an LLM multiple times.

Good for the following:

*   **Parallel prompts**:Create multiple prompts in parallel and do a final pass to merge them.
*   Generating text based on previous generated text e.g., sections within a brief



---



In [19]:
'''
In this example we want to generate a businees name, slogan and then
sales pitch using chain prompting
'''
# prompt to create name for business
name_prompt = [
    {"role": "user", "content": "Create a name for a law firm that leverages LLMs and AI to always win for its clients."}
]
name = pipe(name_prompt, do_sample=True, top_p=0.5)
print(name[0]["generated_text"])

# prompt to create slogan for business
slogan_prompt = [
    {"role": "user", "content": f"Create a slogan for the following law firm '{name}'"}
]
slogan = pipe(slogan_prompt)
print(slogan[0]["generated_text"])

# prompt to create sales pitch for business
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following slogan: '{slogan}'"}
]
output = pipe(sales_prompt)
print(output[0]["generated_text"])

 "AI Advocates Legal Solutions"
 "AI Advocates Legal Solutions: Where Technology Meets Justice."
 "Unlock the power of AI with 'AI Advocates Legal Solutions' – where cutting-edge technology meets the pursuit of justice. Elevate your legal strategy with our innovative approach to legal services."
